# Build a Transformer from Scratch
This notebook teaches the core components of a Transformer with explanations and code.

## 1. Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 2. Self-Attention
Attention computes weighted combinations of token representations.
Formula:
`Attention(Q,K,V)=softmax(QK^T/sqrt(d_k))V`

In [ ]:
def self_attention(x):
    Q=K=V=x
    scores=Q@K.T
    scores/=K.shape[-1]**0.5
    w=F.softmax(scores,dim=-1)
    return w@V


## 3. Single-Head Self Attention Module

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.query=nn.Linear(embed_dim,embed_dim)
        self.key=nn.Linear(embed_dim,embed_dim)
        self.value=nn.Linear(embed_dim,embed_dim)
    def forward(self,x):
        Q=self.query(x);K=self.key(x);V=self.value(x)
        scores=(Q@K.transpose(-2,-1))/(K.size(-1)**0.5)
        w=torch.softmax(scores,dim=-1)
        return w@V


## 4. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embed_dim,num_heads):
        super().__init__()
        self.mha=nn.MultiheadAttention(embed_dim,num_heads,batch_first=True)
    def forward(self,x):
        out,_=self.mha(x,x,x)
        return out


## 5. Feed Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self,e,h):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(e,h),nn.ReLU(),nn.Linear(h,e))
    def forward(self,x): return self.net(x)


## 6. Transformer Block
Residual connections + LayerNorm.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self,e,heads,h):
        super().__init__()
        self.attn=MultiHeadAttention(e,heads)
        self.ff=FeedForward(e,h)
        self.n1=nn.LayerNorm(e); self.n2=nn.LayerNorm(e)
    def forward(self,x):
        x=self.n1(x+self.attn(x))
        x=self.n2(x+self.ff(x))
        return x


## 7. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self,e,max_len=5000):
        super().__init__()
        pe=torch.zeros(max_len,e)
        pos=torch.arange(max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,e,2)*(-math.log(10000.0)/e))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x):
        return x+self.pe[:,:x.size(1)]


## 8. Transformer Encoder

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self,layers,e,heads,h):
        super().__init__()
        self.layers=nn.ModuleList([TransformerBlock(e,heads,h) for _ in range(layers)])
    def forward(self,x):
        for layer in self.layers:
            x=layer(x)
        return x


## 9. Mini GPT Architecture

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self,vocab,e,heads,layers,h):
        super().__init__()
        self.emb=nn.Embedding(vocab,e)
        self.pos=PositionalEncoding(e)
        self.enc=TransformerEncoder(layers,e,heads,h)
        self.fc=nn.Linear(e,vocab)
    def forward(self,x):
        x=self.emb(x)
        x=self.pos(x)
        x=self.enc(x)
        return self.fc(x)


## 10. Example Training Skeleton

In [ ]:
vocab_size=100
model=MiniGPT(vocab_size,64,4,2,128)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)

# x = input_ids
# y = target_ids
# logits=model(x)
# loss=loss_fn(logits.view(-1,vocab_size), y.view(-1))
# loss.backward()
# optimizer.step()
# optimizer.zero_grad()
